# M1 · Tensors, autograd & PyTorch essentials

By the end of this lesson you'll know just enough PyTorch to read and write a training loop, which is all a fine-tune really is.

Work top to bottom and run each cell. When a cell says **Predict**, write your guess down before running it. Then do the exercises in `exercises.py`.

## 1. Tensors

A tensor is an n-dimensional array with a **shape**, a **dtype** and a **device**. Model weights, inputs and gradients are all tensors.

In [ ]:
import torch

x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(x.shape, x.dtype, x.device)

**Predict:** how many bytes does `x` take in memory? (`float32` is 4 bytes per number.)

In [ ]:
print(x.numel(), "numbers ×", x.element_size(), "bytes =", x.numel() * x.element_size(), "bytes")
print("in bfloat16:", x.to(torch.bfloat16).element_size() * x.numel(), "bytes")

That arithmetic is how you'll size models later: an 8-billion-parameter model in bf16 is about 8B × 2 bytes ≈ 16 GB, before training adds anything.

### Broadcasting

When shapes differ, PyTorch stretches size-1 dimensions to match. This is how a bias vector is added to every row of a batch.

In [ ]:
bias = torch.tensor([10.0, 20.0, 30.0])  # shape (3,)
print(x + bias)  # (2, 3) + (3,) -> (2, 3)

**Predict:** what's the shape of `x @ torch.ones(3, 4)`?

In [ ]:
print((x @ torch.ones(3, 4)).shape)

## 2. Autograd

Training means nudging numbers to make a loss smaller. To know which way to nudge, we need gradients. Set `requires_grad=True` and PyTorch records every operation, so `.backward()` can compute them for you.

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
loss = (w - 5) ** 2  # smallest when w = 5
loss.backward()
print("d(loss)/dw =", w.grad)  # 2 * (w - 5) = -4

The gradient is −4: increasing `w` would *decrease* the loss. Gradient descent steps the other way: `w ← w − lr × grad`.

Two rules you'll use constantly:
- Update weights inside `torch.no_grad()`, so the update itself isn't recorded.
- Gradients **accumulate**. Reset them (`w.grad = None`, or `optimizer.zero_grad()`) before the next `backward()`.

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
for _ in range(25):
    loss = (w - 5) ** 2
    loss.backward()
    with torch.no_grad():
        w -= 0.1 * w.grad
    w.grad = None
print(f"w after 25 steps: {w.item():.4f}")

**Predict:** what happens with a learning rate of 1.1 instead of 0.1? Change it above and try.

## 3. Linear regression, by hand

Now a real (tiny) model: fit `y = 2x + 1` from noisy points, with a hand-written training loop. Exercises 1–3 ask you to package these same pieces as functions.

In [ ]:
torch.manual_seed(0)
X = torch.rand(100, 1) * 4
Y = 2 * X + 1 + 0.2 * torch.randn(100, 1)

w = torch.zeros(1, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

for step in range(500):
    pred = X @ w + b
    loss = ((pred - Y) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        for p in (w, b):
            p -= 0.05 * p.grad
            p.grad = None
    if step % 100 == 0:
        print(f"step {step:3d}  loss {loss.item():.4f}")

print(f"learned w = {w.item():.3f}, b = {b.item():.3f}   (true: 2, 1)")

## 4. `nn.Module` and optimizers

Writing every weight by hand doesn't scale. `nn.Linear` holds its own weight and bias, `nn.Module` collects parameters, and an optimizer does the update step. The loop always has the same five beats:

1. forward
2. loss
3. `zero_grad()`
4. `backward()`
5. `step()`

In [ ]:
from torch import nn

model = nn.Linear(1, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.05)

for _ in range(500):
    loss = ((model(X) - Y) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()

print("weight", model.weight.item(), "bias", model.bias.item())
print("parameters:", sum(p.numel() for p in model.parameters()))

## 5. The Mac's GPU: MPS

On Apple Silicon, PyTorch reaches the GPU through the **MPS** device. Tensors and the model must live on the same device.

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("using", device)

big = torch.randn(4096, 4096, device=device)
out = big @ big
print(out.shape, out.device)

Because memory is unified, the GPU uses the same RAM as everything else. Keep an eye on Activity Monitor's memory pressure graph as models get bigger.

## 6. Your task: classify two moons

A straight line can't separate these two classes, so you need a hidden layer with a non-linearity. That's the MLP you'll build.

In [ ]:
import sys

sys.path.insert(0, ".")  # this notebook's folder, for data.py
import matplotlib.pyplot as plt
from data import make_moons

Xm, ym = make_moons(n=1000, noise=0.2)
plt.figure(figsize=(5, 3.5))
plt.scatter(Xm[:, 0], Xm[:, 1], c=ym, cmap="coolwarm", s=8)
plt.title("Two moons: can a network separate them?")
plt.show()

### What to do now

1. Open `exercises.py` and fill in the six exercises.
2. Check them as you go: `make check M=01` (the training test will fail until step 3).
3. Train your network: `uv run python modules/m01_pytorch_essentials/train.py`
4. Run `make check M=01` again. When you reach 95%, M1 turns green on the dashboard.

**Stretch:** re-run `train.py --hidden 4`, then `--hidden 256`, tagging each check with `make check M=01 LABEL="h=4"`. Compare the attempts on the dashboard.